## **Loading the documents**

In [1]:
from ingest import load_faq_data
documents = load_faq_data()

In [2]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

103

In [3]:
documents = documents_llm

In [4]:
doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


The ID becomes the label in our ground truth dataset. We generate questions from a document, so we know that this document holds the answer. Later, search evaluation checks whether search brings back the document with this ID.

This is why every record needs a stable ID. If you can't uniquely identify a document, you can't tell whether search retrieved the right one. When you build your own evaluation set, assign an ID to each record in your knowledge base first.

## Generating questions with structured output

We use an LLM to generate questions for each document.

This is the first time we're using structured output in the course. With structured output, we ask the LLM to return data in a specific format instead of free-form text. For example, instead of getting a paragraph that contains questions, we can ask for a Python object with a questions field.

This is useful when code will process the output. The model returns the same structure every time. We can access the generated questions directly instead of parsing text manually.

We want the output as a list of strings, so we define that structure with a Pydantic model:

In [5]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

The instructions for the LLM:

In [6]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

We ask the LLM to use different wording from the original document. This makes the evaluation more realistic - real users won't phrase their questions the same way as the FAQ.

Call the LLM for one document:

In [2]:
import os
print("OPENAI_API_KEY" in os.environ)  # should print True

True


In [7]:
# from dotenv import load_dotenv
from openai import OpenAI

# load_dotenv()
openai_client = OpenAI()

Prepare the document as JSON:

In [8]:
import json

user_prompt = json.dumps(doc)

Create the messages:

In [9]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

Until now we called responses.create and read response.output_text. For structured output we switch to responses.parse and pass text_format=Questions, which tells the API to return our class instead of free text.

Call the model:

In [10]:
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

The parsed object is available in response.output_parsed:

In [11]:
result = response.output_parsed

print(result)

questions=['Can I still join the course if I found it late?', 'Is it okay to start the course after it has already begun?', 'If I join now, can I still get a certificate?', 'What do I need to do to be eligible for the certificate?', 'Can I submit the project after the submission window closes?']


We can access the list directly:

In [12]:
print(result.questions)

['Can I still join the course if I found it late?', 'Is it okay to start the course after it has already begun?', 'If I join now, can I still get a certificate?', 'What do I need to do to be eligible for the certificate?', 'Can I submit the project after the submission window closes?']


You should see 5 questions that relate to the first FAQ document.

## Reusable utilities

We'll need this pattern again in other evaluation sections today, so we put it in a reusable helper.

It contains helper functions we'll reuse in this module:

* *llm_structured*: calls the OpenAI API with structured output 
* *llm_structured_retry*: retries structured-output calls when a request fails  
* *calc_price*: calculates the price from token usage   
* *calc_total_price*: calculates the total price from multiple usage objects    
* *map_progress*: runs work in parallel and tracks progress. We'll use it in the next lesson.

Import the structured-output helper:

In [13]:
from evaluation_utils import llm_structured

Use it on the same document:

In [14]:
result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

['I just found this course — is it too late to join now?', 'Can I still sign up if I discovered the course after it already started?', 'Am I allowed to join the course late, or do I need to wait for the next run?', 'If I join now, can I still get the certificate somehow?', 'What’s the deadline for the project if I want to be eligible for the certificate?']


## Tracking cost

The response also contains token usage:

In [15]:
usage.input_tokens, usage.output_tokens

(207, 95)

As in the agents module, we calculate the price from response.usage.

Import the price helper:

In [16]:
from evaluation_utils import calc_price

Calculate the cost of this call:

In [17]:
cost = calc_price(usage)

cost

{'input_cost': 0.00015525,
 'output_cost': 0.00042750000000000004,
 'total_cost': 0.00058275}

Now convert these questions into ground truth records:

In [18]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': 'I just found this course — is it too late to join now?',
  'document': '74eb249bbf'},
 {'question': 'Can I still sign up if I discovered the course after it already started?',
  'document': '74eb249bbf'},
 {'question': 'Am I allowed to join the course late, or do I need to wait for the next run?',
  'document': '74eb249bbf'},
 {'question': 'If I join now, can I still get the certificate somehow?',
  'document': '74eb249bbf'},
 {'question': 'What’s the deadline for the project if I want to be eligible for the certificate?',
  'document': '74eb249bbf'}]

Each record has two fields:

* *question*: the question generated by the LLM 
* *document*: the ID of the FAQ document that should answer the question

The document field connects the generated question to the document that contains the answer. Later, when we evaluate search, we'll ask the search engine the generated question. Then we'll check if it retrieves the document with this ID.

We now know how to generate and store questions for one document. In the next lesson, we'll run this for all LLM Zoomcamp FAQ documents and save the full ground truth dataset.

## **Generating Ground Truth for All Documents**

In the previous lesson, we generated questions for one document and converted them into ground truth records.

We want to do the same thing for every document in the FAQ dataset. For each document, we generate questions and save them as ground truth records.

For this part, we'll use *tqdm* for progress bars and *pandas* for saving the final CSV.

If you don't have them installed yet, add them first:

In [ ]:
# uv add tqdm pandas

The processing function takes one document and turns it into ground truth records.

For each document, we:

convert the document to JSON so we can send it to the LLM
ask the LLM to return a *Questions* object
create one ground truth record for each generated question
Each record contains the generated question and the ID of the document that should answer the question.

When we send many requests, one of them might fail. We don't want the entire batch to fail because of one temporary error.

Import the retry helper from *evaluation_utils.py*:

In [ ]:
from evaluation_utils import llm_structured_retry

*llm_structured* makes one structured-output call. *llm_structured_retry* wraps the same call in a retry loop. If one request fails because of a temporary API or network issue, it waits briefly and tries again.

Use it in the processing function:

In [ ]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

Try it for the first 5 documents.

Import *tqdm* and run the loop:

In [ ]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

This works, but it runs one LLM call after another. Running it for all documents this way would take too long.

## Parallel processing
Running the calls one after another wastes most of the time waiting on the network. Each request just sits there until OpenAI responds, so we can fire several at once and wait on them together. We process the documents in parallel and track progress while the requests run.

One caution: don't open too many connections at once, or you'll hit the provider's rate limits. Five or six workers is a safe default here.

Import ThreadPoolExecutor:

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

This submits one job per document, updates the progress bar when a job finishes, and collects the results. If you want a more detailed explanation of ThreadPoolExecutor and futures, ask ChatGPT to walk through this helper line by line.

Then replace the loop with the parallel version:

In [ ]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

*generate_ground_truth* returns two things for each document: the generated records and the token usage.

Split those into separate lists:

In [ ]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

With 5 questions per document, you should get roughly 5x the number of documents.

Calculate the total cost:

In [ ]:
from evaluation_utils import calc_price

total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

total_cost

We'll calculate total cost several times in this module, so the utility file has a helper for it:

In [ ]:
from evaluation_utils import calc_total_price

calc_total_price(usages)

Create a dataframe so we can look at the records as a table and save them as a CSV file.

Create the dataframe:

In [ ]:
import pandas as pd

df_ground_truth = pd.DataFrame(ground_truth)

Because we generated the questions from specific documents, we know which document is correct for each question. We now have the ground truth we need for evaluation.

Save it for later use:

In [ ]:
df_ground_truth.to_csv("data/ground_truth-new.csv", index=False)

We generated this file for the course materials on May 29, 2026. The run used 79 LLM Zoomcamp documents and produced 395 questions.

The FAQ data can change over time. If you run the notebook later, you may see different documents and generated questions. Token usage, cost, and search evaluation results may also change.

The total cost was $0.057187, about 6 cents.

If you don't want to generate the questions yourself, download the file we prepared:

In [ ]:
# PREFIX=https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main

# wget -O data/ground_truth-new.csv ${PREFIX}/04-evaluation/data/ground_truth-new.csv

Now we have questions with known correct documents. In the next lesson, we'll run search for these questions and check whether the correct documents appear in the results.